In [7]:
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [8]:
# -----------------------
# Files
# -----------------------
data_file = "validation_app_type_source.csv"
label_file = "validation_app_type_output.csv"

# -----------------------
# Load source data
# -----------------------
ldn_filtr = pd.read_csv(data_file)

In [9]:
ldn_filtr.shape

(486, 216)

In [10]:
# Ensure decision_date is datetime (safe even if already)
if "decision_date" in ldn_filtr.columns:
    ldn_filtr["decision_date"] = pd.to_datetime(ldn_filtr["decision_date"], errors="coerce")

# -----------------------
# Create output file if missing
# -----------------------
if not os.path.exists(label_file):
    pd.DataFrame({"lpa_app_no": [], "description": [], "is_full_planning": []}).to_csv(label_file, index=False)

# Load existing validations
validated = pd.read_csv(label_file)
validated_set = set(validated["lpa_app_no"].dropna().astype(str))

# Make sure IDs are comparable as strings
ldn_filtr["lpa_app_no"] = ldn_filtr["lpa_app_no"].astype(str)
validated_set = set(map(str, validated_set))

# -----------------------
# Filter rows to validate:
#   - is_full_planning is NA
#   - not already validated
# -----------------------
to_validate = ldn_filtr[ldn_filtr["is_full_planning"].isna()].copy()
to_validate = to_validate[~to_validate["lpa_app_no"].isin(validated_set)].copy()

rows = to_validate.to_dict("records")

# -----------------------
# Widgets + state
# -----------------------
output = widgets.Output()
progress = widgets.HTML()

true_btn = widgets.Button(description="✅ True", button_style="success")
false_btn = widgets.Button(description="❌ False", button_style="danger")
skip_btn = widgets.Button(description="⏭️ Skip", button_style="warning")

index_tracker = {"idx": 0}  # mutable index


def append_validation(app_no: str, desc: str, value):
    """Append one validated row to CSV immediately."""
    df = pd.DataFrame([{
        "lpa_app_no": app_no,
        "description": desc,
        "is_full_planning": value
    }])
    df.to_csv(label_file, mode="a", header=False, index=False)


def show_row():
    idx = index_tracker["idx"]

    # Progress header
    progress.value = f"<b>Row {min(idx+1, len(rows))} / {len(rows)}</b>"

    if idx >= len(rows):
        with output:
            clear_output()
            display(HTML("<h3>🎉 All eligible NA rows have been validated (or skipped).</h3>"))
        return

    row = rows[idx]

    app_no = str(row.get("lpa_app_no", ""))
    desc = row.get("description", "")
    lpa_name = row.get("lpa_name", "")
    decision_date = row.get("decision_date", "")

    # Nicely format decision_date if it's a Timestamp/str
    try:
        decision_date_str = pd.to_datetime(decision_date).strftime("%Y-%m-%d")
    except Exception:
        decision_date_str = str(decision_date)

    with output:
        clear_output()
        display(HTML(f"""
        <div style="margin-bottom:10px;">
            <div><b>Application ID:</b> {app_no}</div>
            <div><b>LPA name:</b> {lpa_name}</div>
            <div><b>Decision date:</b> {decision_date_str}</div>
        </div>

        <b>DESCRIPTION:</b>
        <div style="max-height:300px; overflow:auto; border:1px solid #ccc; padding:8px; white-space:pre-wrap;">
            {"" if pd.isna(desc) else str(desc)}
        </div>
        """))


def on_true(_):
    idx = index_tracker["idx"]
    row = rows[idx]
    append_validation(str(row["lpa_app_no"]), row.get("description", ""), True)
    index_tracker["idx"] += 1
    show_row()


def on_false(_):
    idx = index_tracker["idx"]
    row = rows[idx]
    append_validation(str(row["lpa_app_no"]), row.get("description", ""), False)
    index_tracker["idx"] += 1
    show_row()


def on_skip(_):
    # Skip = don't write anything
    index_tracker["idx"] += 1
    show_row()


true_btn.on_click(on_true)
false_btn.on_click(on_false)
skip_btn.on_click(on_skip)

# -----------------------
# Display UI
# -----------------------
display(progress)
display(widgets.HBox([true_btn, false_btn, skip_btn]))
display(output)

show_row()

HTML(value='')

Output()